## Cambiar Permisos por Carpeta en /Shared

**Objetivo:** Aplicar permisos granulares a carpetas específicas dentro de `/Shared`.

**Requisitos:**
- Ejecutar con un usuario que sea **Workspace Admin**
- Definir en el diccionario `FOLDERS_CONFIG` las carpetas y sus permisos

**Niveles de permiso:**
| Nivel | Puede ver | Puede ejecutar | Puede editar | Puede crear/borrar/mover | Puede cambiar permisos |
|-------|-----------|----------------|--------------|--------------------------|------------------------|
| CAN_READ | ✓ | | | | |
| CAN_RUN | ✓ | ✓ | | | |
| CAN_EDIT | ✓ | ✓ | ✓ | | |
| CAN_MANAGE | ✓ | ✓ | ✓ | ✓ | ✓ |

**Flujo:**
1. Definir configuración de carpetas y permisos
2. Resolver los IDs de cada carpeta automáticamente
3. Ver permisos actuales de cada carpeta
4. Aplicar los nuevos permisos
5. Verificar cambios

In [ ]:
import requests

# ── Derivar host y token del contexto (evita el "Invalid access to Org") ──
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()          # ← host correcto del workspace actual
token = ctx.apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Sanity check rápido antes de seguir
_r = requests.get(f"{host}/api/2.0/workspace/get-status",
                  headers=headers, params={"path": "/Shared"})
print(f"Host: {host}")
print(f"Test /Shared → {_r.status_code}: {_r.json().get('object_type', _r.json().get('message'))}")

# ===============================================================================
# CONFIGURACION DE CARPETAS Y PERMISOS
#
# Rutas verificadas contra el workspace el 2026-07-30 con la celda de diagnostico.
# OJO: en este workspace NO existe ninguna carpeta *_prd, solo *_dev.
#
# /Shared (raiz) quedo fuera a proposito: la API devuelve
#   400 Cannot modify permissions of directory  -- su ACL es fija por diseno.
# ===============================================================================

FOLDERS_CONFIG = {
    # --- Bundle de Arquitectura de Datos (dev) ---
    "/Shared/databricks_repo_ad_dev": {
        "CAN_MANAGE": [
            {"group_name": "admins"},
            {"group_name": "GS_ADMINISTRADORCATALOGO_CORONA"},
            {"group_name": "GS_ARQUITECTODATOS_CORONA"},
            {"service_principal_name": "sp-databricks-prd-contributor-arqanalitica"},
        ],
        "CAN_EDIT": [
            {"group_name": "GS_ANALISTADATOS_CORONA"},
            {"group_name": "GS_CIENTIFICODATOS_CORONA"},
        ],
        "CAN_READ": [
            {"group_name": "users"},
        ],
    },

    # --- Bundle de Analitica/Ingenieria (dev) ---
    "/Shared/databricks_repo_aq_dev": {
        "CAN_MANAGE": [
            {"group_name": "admins"},
            {"group_name": "GS_ADMINISTRADORCATALOGO_CORONA"},
            {"group_name": "GS_ARQUITECTODATOS_CORONA"},
            {"service_principal_name": "sp-databricks-prd-contributor-arqanalitica"},
        ],
        "CAN_EDIT": [
            {"group_name": "GS_INGENIERODATOS_CORONA"},
            {"group_name": "GS_INGENIERODATOS_IDATA"},
        ],
        "CAN_READ": [
            {"group_name": "users"},
        ],
    },

    # --- Pendientes de definir: descomenta cuando sepas que grupos van ---
    # "/Shared/databricks_repo_gb_dev": {...},   # id 772919698670255
    # "/Shared/databricks_repo_dev":    {...},   # id 772919698669634
    # "/Shared/databricks_aq":          {...},   # OJO: es REPO, no DIRECTORY
}

print(f"Configuracion definida para {len(FOLDERS_CONFIG)} objeto(s):")
for path in FOLDERS_CONFIG:
    print(f"   - {path}")


### 🔎 Explorar el workspace (diagnostico)

Las rutas de `FOLDERS_CONFIG` fallaron con `Path doesn't exist`. Esta celda lista
lo que hay realmente en `/Shared` y `/Repos` para encontrar los paths correctos.

Solo lee: no modifica nada.


In [ ]:
def listar(path, nivel=0, max_nivel=1):
    """Lista recursivamente el contenido de una ruta del workspace."""
    resp = requests.get(
        f"{host}/api/2.0/workspace/list",
        headers=headers,
        params={"path": path}
    )
    if resp.status_code != 200:
        print(f"{'  ' * nivel}\u274c {path}: {resp.json().get('message', resp.text)}")
        return

    objetos = sorted(resp.json().get("objects", []), key=lambda o: o["path"])
    if not objetos:
        print(f"{'  ' * nivel}(vacio)")
        return

    iconos = {"DIRECTORY": "\U0001f4c1", "REPO": "\U0001f517", "NOTEBOOK": "\U0001f4d3", "FILE": "\U0001f4c4"}
    for obj in objetos:
        tipo = obj["object_type"]
        icono = iconos.get(tipo, "\u2753")
        oid = obj.get("object_id", "-")
        print(f"{'  ' * nivel}{icono} {obj['path']:<55} [{tipo:<9}] id={oid}")
        # Solo bajamos por DIRECTORY; un REPO se trata como una unidad
        if tipo == "DIRECTORY" and nivel < max_nivel:
            listar(obj["path"], nivel + 1, max_nivel)


for raiz in ["/Shared", "/Repos"]:
    print("\u2550" * 90)
    print(f" CONTENIDO DE {raiz}")
    print("\u2550" * 90)
    listar(raiz)
    print()

print("\u2550" * 90)
print("Copia los paths exactos que necesites a FOLDERS_CONFIG y re-ejecuta la celda anterior.")
print("Ojo con el object_type: DIRECTORY -> /permissions/directories/{id}")
print("                        REPO      -> /permissions/repos/{id}")


In [ ]:
# Endpoint de permisos segun el tipo de objeto
ENDPOINT_POR_TIPO = {
    "DIRECTORY": "directories",
    "REPO": "repos",
    "NOTEBOOK": "notebooks",
}


def resolve_object(path):
    """Devuelve (object_id, object_type) de un path del workspace, o (None, None)."""
    resp = requests.get(
        f"{host}/api/2.0/workspace/get-status",
        headers=headers,
        params={"path": path}
    )
    if resp.status_code != 200:
        print(f"  ERROR obteniendo {path}: {resp.json().get('message', resp.text)}")
        return None, None

    data = resp.json()
    tipo = data.get("object_type")
    if tipo not in ENDPOINT_POR_TIPO:
        print(f"  {path} es {tipo}: no soporta ACLs por este metodo")
        return None, None

    return str(data["object_id"]), tipo


def permissions_url(object_id, object_type):
    """URL del endpoint de permisos correspondiente al tipo de objeto."""
    return f"{host}/api/2.0/permissions/{ENDPOINT_POR_TIPO[object_type]}/{object_id}"


# Resolver todos los objetos configurados
objetos = {}   # path -> (object_id, object_type)
print("=" * 80)
print(" RESOLVIENDO OBJETOS DEL WORKSPACE")
print("=" * 80)
for path in FOLDERS_CONFIG:
    oid, tipo = resolve_object(path)
    if oid:
        objetos[path] = (oid, tipo)
        print(f"  OK  {path:<45} {tipo:<10} id={oid}")
    else:
        print(f"  --  {path:<45} NO RESUELTO")

print(f"\n{len(objetos)}/{len(FOLDERS_CONFIG)} objetos resueltos")


In [ ]:
def show_current_permissions(path, object_id, object_type):
    """Muestra los permisos actuales de un objeto del workspace."""
    resp = requests.get(permissions_url(object_id, object_type), headers=headers)
    if resp.status_code != 200:
        print(f"  ERROR leyendo permisos: {resp.json()}")
        return

    print(f"\n{'-' * 80}")
    print(f" {path}  [{object_type}] id={object_id}")
    print(f"{'-' * 80}")
    print(f"  {'Grupo/Entidad':<45} {'Permiso':<15} {'Herencia'}")
    print(f"  {'-' * 72}")

    for acl in resp.json().get("access_control_list", []):
        name = (acl.get("group_name")
                or acl.get("user_name")
                or acl.get("service_principal_name")
                or "???")
        perms = acl.get("all_permissions") or []
        if not perms:
            print(f"  {name:<45} {'(sin permisos)':<15} -")
            continue
        for p in perms:
            level = p.get("permission_level", "N/A")
            herencia = "heredado" if p.get("inherited", False) else "directo"
            print(f"  {name:<45} {level:<15} {herencia}")


print("=" * 80)
print(" PERMISOS ACTUALES (ANTES DEL CAMBIO)")
print("=" * 80)
for path, (oid, tipo) in objetos.items():
    show_current_permissions(path, oid, tipo)


In [ ]:
def build_acl(config):
    """Construye el access_control_list a partir de la configuracion de un objeto."""
    acl = []
    for permission_level, entities in config.items():
        for entity in entities:
            entry = dict(entity)   # copia para no mutar el original
            entry["permission_level"] = permission_level
            acl.append(entry)
    return acl


def apply_permissions(path, object_id, object_type, config, dry_run=True):
    """Aplica permisos a un objeto. Con dry_run=True solo muestra lo que haria."""
    acl = build_acl(config)

    print(f"\n{'-' * 80}")
    print(f" {path}  [{object_type}] id={object_id}")
    print(f"{'-' * 80}")

    if dry_run:
        print("  DRY-RUN (no se aplican cambios):")
        for entry in acl:
            name = (entry.get("group_name")
                    or entry.get("user_name")
                    or entry.get("service_principal_name"))
            print(f"     {name:<45} -> {entry['permission_level']}")
        return True

    resp = requests.put(
        permissions_url(object_id, object_type),
        headers=headers,
        json={"access_control_list": acl},
    )
    if resp.status_code == 200:
        print("  Permisos aplicados exitosamente")
        return True

    print(f"  ERROR ({resp.status_code}): {resp.json().get('message', resp.text)}")
    return False


# ===============================================================================
# PUT REEMPLAZA LA ACL COMPLETA: cualquier grupo que hoy tenga acceso y no este
# en FOLDERS_CONFIG lo pierde. Revisa el dry-run antes de poner DRY_RUN = False.
#
# Estas carpetas son despliegues de Asset Bundles: el pipeline puede sobrescribir
# estos permisos en el proximo deploy. Para permisos permanentes usa el bloque
# permissions: del databricks.yml de cada bundle.
# ===============================================================================

DRY_RUN = True   # <- cambiar a False para aplicar de verdad

print("=" * 80)
print(" MODO DRY-RUN: solo se muestra lo que se aplicaria" if DRY_RUN
      else " MODO EJECUCION: se aplicaran los cambios")
print("=" * 80)

ok = 0
for path, (oid, tipo) in objetos.items():
    if apply_permissions(path, oid, tipo, FOLDERS_CONFIG[path], dry_run=DRY_RUN):
        ok += 1

print(f"\n{'=' * 80}")
if DRY_RUN:
    print(f"Dry-run completo: {ok}/{len(objetos)} objetos simulados")
    print("Para aplicar de verdad, cambia DRY_RUN = False y re-ejecuta esta celda")
else:
    print(f"{ok}/{len(objetos)} objetos actualizados exitosamente")


In [ ]:
print("=" * 80)
print(" VERIFICACION: PERMISOS DESPUES DEL CAMBIO")
print("=" * 80)
for path, (oid, tipo) in objetos.items():
    show_current_permissions(path, oid, tipo)

print(f"\n{'=' * 80}")
print("Verificacion completa")
